In [ ]:
!pip install -q groq requests

In [ ]:
import json
import unicodedata
import requests
from groq import Groq
from google.colab import userdata

# ── Configuração ──────────────────────────────────────────────────────────────

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
TMDB_API_KEY = userdata.get('TMDB_API_KEY')

client = Groq(api_key=GROQ_API_KEY)

TMDB_IMAGE_BASE = "https://image.tmdb.org/t/p/w342"

GENRES = {
    16: "Animação",   35: "Comédia",        10751: "Família",
    14: "Fantasia",   18: "Drama",           10749: "Romance",
    99: "Documentário", 12: "Aventura",      28: "Ação",
    27: "Terror",    878: "Ficção Científica", 80: "Crime",
  9648: "Mistério",   53: "Suspense",       10752: "Guerra",
    37: "Faroeste",   36: "História",        10402: "Música",
 10770: "Telefilme",
}

STOPWORDS = {
    "o", "a", "os", "as", "um", "uma", "de", "do", "da", "dos", "das",
    "e", "em", "no", "na", "nos", "nas", "por", "para", "com", "que",
    "the", "an", "of", "in", "on", "at", "to", "and", "is", "it",
}

# ── Utilitários ───────────────────────────────────────────────────────────────
def normalizar(texto: str) -> str:
    sem_acento = unicodedata.normalize("NFD", texto)
    sem_acento = "".join(c for c in sem_acento if unicodedata.category(c) != "Mn")
    return " ".join(sem_acento.lower().strip().split())

def palavras_chave(texto: str) -> list[str]:
    return [p for p in normalizar(texto).split() if len(p) > 3 and p not in STOPWORDS]


# ── IA ────────────────────────────────────────────────────────────────────────
def obter_filmes_da_ia(prompt_usuario: str, historico: list[str]) -> dict:
    proibidos = ""
    if historico:
        proibidos = (
            "\n\n### FILMES PROIBIDOS:\n"
            + "\n".join(f"- {f}" for f in historico)
        )

    sistema = (
        "Você é um especialista em cinema. Sua UNICA função é indicar filmes REAIS "
        "que existam no IMDb e no TMDB com mais de 50.000 avaliações de usuários.\n\n"
        "### REGRAS ABSOLUTAS:\n"
        "1. NUNCA invente títulos.\n"
        "2. NUNCA altere o ano de lançamento.\n"
        "3. NUNCA combine partes de títulos diferentes.\n"
        "4. SEMPRE use a grafia correta do título oficial.\n"
        "5. Use apenas filmes com nota IMDb >= 7.0 e mais de 50.000 votos.\n"
        "6. NUNCA repita filmes proibidos.\n"
        "7. Sugira EXATAMENTE 3 filmes.\n"
        "8. Os filmes DEVEM combinar com o pedido do usuário.\n\n"
        "### FORMATO OBRIGATORIO (apenas JSON):\n"
        '{"mensagem": "frase curta e animada (max 10 palavras)", '
        '"lista_filmes": ["Título 1", "Título 2", "Título 3"]}'
        + proibidos
    )

    resposta = client.chat.completions.create(
        messages=[
            {"role": "system", "content": sistema},
            {"role": "user",   "content": prompt_usuario},
        ],
        model="llama-3.3-70b-versatile",
        temperature=0.6,
        response_format={"type": "json_object"},
    )

    dados = json.loads(resposta.choices[0].message.content)
    if "lista_filmes" not in dados or not isinstance(dados["lista_filmes"], list):
        raise ValueError(f"Resposta fora do formato esperado: {dados}")
    return dados


# ── TMDB ──────────────────────────────────────────────────────────────────────
def _buscar_validado(nome: str) -> dict | None:
    url = (
        "https://api.themoviedb.org/3/search/movie"
        f"?api_key={TMDB_API_KEY}&query={requests.utils.quote(nome)}&language=pt-BR"
    )
    try:
        resultados = requests.get(url, timeout=8).json().get("results", [])[:5]
        chaves = palavras_chave(nome)
        candidatos = [
            f for f in resultados
            if f.get("vote_count", 0) >= 500
            and (not chaves or any(
                p in normalizar(f.get("title", "")) or
                p in normalizar(f.get("original_title", ""))
                for p in chaves
            ))
        ]
        return max(candidatos, key=lambda f: f.get("vote_count", 0)) if candidatos else None
    except Exception:
        return None

def _corrigir_grafia(nome: str) -> str:
    url = (
        "https://api.themoviedb.org/3/search/movie"
        f"?api_key={TMDB_API_KEY}&query={requests.utils.quote(nome)}&language=en-US"
    )
    try:
        chaves = palavras_chave(nome)
        for filme in requests.get(url, timeout=8).json().get("results", [])[:3]:
            if filme.get("vote_count", 0) < 10_000:
                continue
            titulo_orig = normalizar(filme.get("original_title", ""))
            if chaves and any(p in titulo_orig for p in chaves):
                return filme.get("original_title", nome)
    except Exception:
        pass
    return nome

def buscar_detalhes_tmdb(nome: str) -> dict | None:
    resultado = _buscar_validado(nome)
    if resultado:
        return resultado
    corrigido = _corrigir_grafia(nome)
    if normalizar(corrigido) != normalizar(nome):
        return _buscar_validado(corrigido)
    return None

def buscar_streaming(movie_id: int) -> list[str]:
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/watch/providers?api_key={TMDB_API_KEY}"
    try:
        br = requests.get(url, timeout=8).json().get("results", {}).get("BR", {})
        if br.get("flatrate"):
            return [p["provider_name"] for p in br["flatrate"]]
        if br.get("rent"):
            return ["Aluguel: " + ", ".join(p["provider_name"] for p in br["rent"])]
        return ["Não disponível no BR"]
    except Exception:
        return ["Erro ao buscar"]

def get_poster_url(poster_path):
    if poster_path:
        return f"{TMDB_IMAGE_BASE}{poster_path}"
    return "https://placehold.co/342x513/1a0000/666?text=Sem+Capa"

print("Funções carregadas com sucesso!")

In [ ]:
import gradio as gr

historico_global = []
historico_norm_global = set()

# ── HTML dos Cards ────────────────────────────────────────────────────────────

def gerar_card_filme(filme: dict) -> str:
    poster_url  = get_poster_url(filme.get("poster_path"))
    titulo      = filme.get("title", "Desconhecido")
    ano         = filme.get("release_date", "????")[:4]
    nota        = filme.get("vote_average", 0)
    votos       = filme.get("vote_count", 0)
    overview    = filme.get("overview", "Sem sinopse disponível.")
    overview    = overview[:230] + "..." if len(overview) > 230 else overview
    generos     = [GENRES.get(g, "Geral") for g in filme.get("genre_ids", [])[:3]]
    generos_str = "  ·  ".join(generos) if generos else "Geral"
    streams     = buscar_streaming(filme["id"])
    streams_str = "  ·  ".join(streams)

    estrelas = "★" * round(nota / 2) + "☆" * (5 - round(nota / 2))

    return f"""
<div style="
  display: flex;
  gap: 20px;
  background: linear-gradient(135deg, rgba(20,0,0,0.95) 0%, rgba(40,5,5,0.9) 100%);
  border: 1px solid rgba(229,9,20,0.25);
  border-left: 3px solid #e50914;
  border-radius: 6px;
  padding: 16px;
  margin: 10px 0;
  box-shadow: 0 4px 20px rgba(229,9,20,0.1), 0 2px 8px rgba(0,0,0,0.6);
  font-family: 'Bebas Neue', 'Georgia', serif;
  transition: transform 0.2s;
">
  <div style="flex-shrink:0; position:relative;">
    <img
      src="{poster_url}"
      alt="{titulo}"
      style="
        width: 100px;
        height: 150px;
        object-fit: cover;
        border-radius: 4px;
        box-shadow: 0 4px 16px rgba(0,0,0,0.8), 0 0 0 1px rgba(229,9,20,0.2);
        display: block;
      "
      onerror="this.src='https://placehold.co/100x150/1a0000/555?text=Sem+Capa'"
    />
    <div style="
      position: absolute;
      bottom: 4px; left: 4px; right: 4px;
      background: rgba(229,9,20,0.85);
      color: white;
      font-size: 11px;
      font-weight: 700;
      text-align: center;
      border-radius: 0 0 3px 3px;
      padding: 2px 0;
      letter-spacing: 1px;
    ">{ano}</div>
  </div>

  <div style="flex:1; color:#e5e5e5; min-width:0;">
    <div style="
      font-size: 17px;
      font-weight: 800;
      color: #ffffff;
      letter-spacing: 0.5px;
      margin-bottom: 5px;
      text-transform: uppercase;
      font-family: 'Bebas Neue', 'Impact', sans-serif;
    ">{titulo}</div>

    <div style="display:flex; gap:6px; flex-wrap:wrap; margin-bottom:8px; align-items:center;">
      <span style="color:#e50914; font-size:13px; font-weight:700; letter-spacing:1px;">{estrelas}</span>
      <span style="color:#ccc; font-size:12px;">{nota:.1f}/10</span>
      <span style="color:#666; font-size:11px;">({votos:,} votos)</span>
    </div>

    <div style="margin-bottom:7px;">
      <span style="
        font-size: 11px;
        color: #e50914;
        text-transform: uppercase;
        letter-spacing: 1.5px;
        font-weight: 600;
      ">{generos_str}</span>
    </div>

    <p style="
      font-size: 12px;
      color: #aaa;
      line-height: 1.55;
      margin: 0 0 8px 0;
      font-family: -apple-system, 'Helvetica Neue', sans-serif;
      font-weight: 300;
    ">{overview}</p>

    <div style="
      border-top: 1px solid rgba(229,9,20,0.15);
      padding-top: 7px;
      font-size: 11px;
      color: #888;
      font-family: -apple-system, sans-serif;
    ">▶ <span style="color:#e50914; font-weight:600;">Onde assistir:</span> {streams_str}</div>
  </div>
</div>
"""


# ── Lógica do chat ────────────────────────────────────────────────────────────

def processar_mensagem(mensagem: str, historico_chat: list) -> tuple:
    global historico_global, historico_norm_global

    if not mensagem.strip():
        return historico_chat, ""

    historico_chat.append({"role": "user", "content": mensagem})

    try:
        sugestao = obter_filmes_da_ia(mensagem, historico_global)

        html = f"""
<div style="font-family:-apple-system,sans-serif; padding:4px 0;">
  <div style="
    font-size:14px; font-weight:600; color:#e5e5e5;
    border-bottom: 1px solid rgba(229,9,20,0.3);
    padding-bottom: 10px;
    margin-bottom: 12px;
    letter-spacing: 0.5px;
  ">🎬 {sugestao['mensagem']}</div>
"""

        exibidos = 0
        for nome in sugestao["lista_filmes"]:
            if normalizar(nome) in historico_norm_global:
                continue

            historico_global.append(nome)
            historico_norm_global.add(normalizar(nome))

            filme = buscar_detalhes_tmdb(nome)
            if not filme:
                html += f"<p style='color:#e67e22; font-size:12px;'>⚠ '{nome}' não encontrado no TMDB.</p>"
                continue

            html += gerar_card_filme(filme)
            exibidos += 1

        if exibidos == 0:
            html += "<p style='color:#e74c3c;'>⚠ Nenhum filme validado. Tente outro pedido.</p>"
        else:
            html += f"""
<div style="
  margin-top:12px; font-size:11px; color:#555;
  text-align:right; font-family:monospace;
">📋 {len(historico_global)} filme(s) no histórico</div>
"""
        html += "</div>"
        historico_chat.append({"role": "assistant", "content": html})

    except json.JSONDecodeError:
        historico_chat.append({"role": "assistant", "content": "<p style='color:#e74c3c;'>❌ Formato inválido. Tente novamente.</p>"})
    except Exception as e:
        historico_chat.append({"role": "assistant", "content": f"<p style='color:#e74c3c;'>❌ Erro: {str(e)}</p>"})

    return historico_chat, ""


def limpar_historico():
    global historico_global, historico_norm_global
    historico_global = []
    historico_norm_global = set()
    return [], ""


# ── CSS / Tema ────────────────────────────────────────────────────────

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Bebas+Neue&display=swap');

body, .gradio-container {
    background: #141414 !important;
}

.gradio-container {
    max-width: 860px !important;
    margin: 0 auto !important;
    background: transparent !important;
}

/* Fundo degradê vermelho-preto como na imagem */
gradio-app {
    background: radial-gradient(ellipse at 70% 30%, #8b0000 0%, #2d0000 40%, #0d0d0d 75%, #000 100%) !important;
    min-height: 100vh;
}

/* Remove bordas do container principal */
.contain { background: transparent !important; }

/* Card de input */
.input-card {
    background: rgba(20,5,5,0.85) !important;
    border: 1px solid rgba(229,9,20,0.2) !important;
    border-radius: 8px !important;
    padding: 24px !important;
    backdrop-filter: blur(10px);
}

/* Textbox */
textarea, input[type=text] {
    background: rgba(30,5,5,0.9) !important;
    border: 1px solid rgba(229,9,20,0.3) !important;
    border-radius: 6px !important;
    color: #e5e5e5 !important;
    font-family: 'Helvetica Neue', sans-serif !important;
    font-size: 14px !important;
    caret-color: #e50914;
}
textarea:focus, input[type=text]:focus {
    border-color: rgba(229,9,20,0.7) !important;
    box-shadow: 0 0 0 2px rgba(229,9,20,0.15) !important;
    outline: none !important;
}
textarea::placeholder, input::placeholder {
    color: #555 !important;
}

/* Botão principal - vermelho Netflix */
button.primary {
    background: #e50914 !important;
    color: white !important;
    border: none !important;
    border-radius: 4px !important;
    font-family: 'Bebas Neue', 'Impact', sans-serif !important;
    font-size: 15px !important;
    letter-spacing: 2px !important;
    padding: 12px 28px !important;
    cursor: pointer !important;
    transition: background 0.2s, transform 0.1s !important;
    text-transform: uppercase !important;
    width: 100% !important;
}
button.primary:hover {
    background: #f40612 !important;
    transform: translateY(-1px);
    box-shadow: 0 4px 20px rgba(229,9,20,0.4) !important;
}
button.primary:active { transform: translateY(0); }

/* Botão secundário */
button.secondary {
    background: rgba(229,9,20,0.12) !important;
    color: #e50914 !important;
    border: 1px solid rgba(229,9,20,0.35) !important;
    border-radius: 4px !important;
    font-size: 12px !important;
    letter-spacing: 1px !important;
    transition: all 0.2s !important;
}
button.secondary:hover {
    background: rgba(229,9,20,0.25) !important;
    border-color: #e50914 !important;
}

/* Chatbot */
.chatbot, .message-wrap {
    background: transparent !important;
    border: none !important;
}
.chatbot .wrap {
    background: transparent !important;
}

/* Mensagem do usuário */
.chatbot .user .message {
    background: rgba(229,9,20,0.15) !important;
    border: 1px solid rgba(229,9,20,0.25) !important;
    border-radius: 8px 8px 2px 8px !important;
    color: #e5e5e5 !important;
    font-size: 13px !important;
}

/* Mensagem do bot */
.chatbot .bot .message {
    background: rgba(10,0,0,0.7) !important;
    border: 1px solid rgba(229,9,20,0.1) !important;
    border-radius: 8px 8px 8px 2px !important;
    padding: 12px !important;
}

/* Labels */
label span, .label-wrap span {
    color: #777 !important;
    font-size: 11px !important;
    text-transform: uppercase;
    letter-spacing: 1px;
}

footer { display: none !important; }
svg.feather-copy { color: #e50914 !important; }
"""


# ── Interface ─────────────────────────────────────────────────────────────────

with gr.Blocks(
    title="Cinema Emocional",
    css=CSS,
    theme=gr.themes.Base().set(
        body_background_fill="#141414",
        block_background_fill="transparent",
        block_border_color="transparent",
        input_background_fill="#1a0000",
        button_primary_background_fill="#e50914",
        button_primary_background_fill_hover="#f40612",
        button_secondary_background_fill="transparent",
        color_accent="#e50914",
        color_accent_soft="rgba(229,9,20,0.15)",
    )
) as interface:

    # ── Header ──
    gr.HTML("""
    <div style="text-align:center; padding:36px 0 24px; font-family:-apple-system,sans-serif;">
      <div style="display:inline-flex; align-items:center; gap:12px; margin-bottom:10px;">
        <div style="
          width:52px; height:52px;
          background: radial-gradient(circle, #e50914 0%, #8b0000 100%);
          border-radius:50%;
          display:flex; align-items:center; justify-content:center;
          font-size:26px;
          box-shadow: 0 0 20px rgba(229,9,20,0.5);
        ">🤖</div>
        <span style="
          font-family:'Bebas Neue','Impact',sans-serif;
          font-size:3rem;
          letter-spacing:3px;
          background: linear-gradient(135deg, #ffffff 0%, #e50914 60%, #8b0000 100%);
          -webkit-background-clip: text;
          -webkit-text-fill-color: transparent;
          background-clip: text;
          line-height:1;
        ">CinemaEmocional</span>
      </div>
      <p style="color:#888; font-size:14px; margin:0; letter-spacing:0.5px;">
        Seu assistente pessoal para encontrar o filme perfeito
      </p>
    </div>
    """)

    # ── Input Card ──
    with gr.Group(elem_classes="input-card"):
        msg = gr.Textbox(
            label="",
            placeholder="Ex: um filme de ação",
            lines=3,
            max_lines=5,
            show_label=False,
        )

        gr.HTML("""
        <div style="padding:10px 0 14px; font-family:-apple-system,sans-serif;">
          <p style="color:#e50914; font-size:12px; font-weight:700; letter-spacing:1px; margin:0 0 6px; text-transform:uppercase;">Exemplos:</p>
          <ul style="margin:0; padding-left:16px; color:#666; font-size:12px; line-height:1.9;">
            <li>Quero algo engraçado para relaxar depois do trabalho</li>
            <li>Estou procurando um thriller que me deixe na ponta da cadeira</li>
            <li>Algo romântico para assistir com minha namorada</li>
          </ul>
        </div>
        """)

        enviar = gr.Button("▶  Encontrar Filmes Perfeitos", variant="primary")

    # ── Seção de resultados ──
    gr.HTML("""
    <div style="
      text-align:center;
      padding: 28px 0 14px;
      font-family:'Bebas Neue','Impact',sans-serif;
      font-size:1.6rem;
      letter-spacing:3px;
      color:#e5e5e5;
    ">Filmes perfeitos pra você</div>
    """)

    chatbot = gr.Chatbot(
        label="",
        type="messages",
        height=580,
        show_copy_button=True,
        render_markdown=False,
        sanitize_html=False,
        show_label=False,
        placeholder="<div style='text-align:center; color:#333; padding:60px 20px; font-family:-apple-system,sans-serif;'><div style='font-size:40px; margin-bottom:12px;'>🎬</div><div style='font-size:14px; letter-spacing:1px;'>Aguardando seu pedido...</div></div>",
    )

    with gr.Row():
        limpar = gr.Button("🗑  Limpar histórico e recomeçar", variant="secondary", size="sm")

    # Eventos
    msg.submit(processar_mensagem, [msg, chatbot], [chatbot, msg])
    enviar.click(processar_mensagem, [msg, chatbot], [chatbot, msg])
    limpar.click(limpar_historico, None, [chatbot, msg])

interface.launch(share=False, debug=False)

/tmp/ipykernel_255/635774721.py:310: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_255/635774721.py:310: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_255/635774721.py:390: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel_255/635774721.py:390: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>